# Genetic Algorithm for Multiple Sequence Alignment (MSA)

In [1]:
%run fitness_function.ipynb

import random
from copy import deepcopy
from matplotlib import pyplot as plt

Score1 105.0
Score2: 67.0
Score3: -25.0


## Individual Class


In [2]:
class Individual:
    def __init__(self, sequences, max_gaps=5):
        self.original_sequences = sequences
        self.max_gaps = max_gaps
        self.gap_positions = self.random_gap_positions()
        self.code = self.build_alignment()
        self.calc_fitness()
    
    # Generate random number and random positions for gaps for each sequence in alignment
    def random_gap_positions(self):
        gap_positions = []
        for seq in self.original_sequences:
            num_gaps = random.randint(0, self.max_gaps)
            total_len = len(seq) + num_gaps
            if num_gaps > 0:
                positions = sorted(random.sample(range(total_len), num_gaps))
            else:
                positions = []
            gap_positions.append(positions)
        return gap_positions
   
    # Create alignment with given sequences and gap positions
    def build_alignment(self):
        alignment = []
        
        for seq, gap_positions in zip(self.original_sequences, self.gap_positions):
            total_len = len(seq) + len(gap_positions)
            
            result = []
            seq_idx = 0
            gap_set = set(gap_positions)
            
            for i in range(total_len):
                if i in gap_set:
                    result.append('-')
                else:
                    if seq_idx < len(seq):
                        result.append(seq[seq_idx])
                        seq_idx += 1
            
            alignment.append(''.join(result))
        
        max_len = max(len(s) for s in alignment)
        alignment = [s + '-' * (max_len - len(s)) for s in alignment]
        return alignment
    
    def calc_fitness(self):
        self.fitness = calculate_sp_score(self.code)

In [3]:
def print_individual(alignment: Individual):
    print(f"Score: {alignment.fitness}")
    for seq in alignment.code:
        print(f"  {seq}")

## Selection

In [4]:
def selection(population, tournament_size):
    k = min(len(population), tournament_size)
    participants = random.sample(population, k)
    return max(participants, key=lambda x: x.fitness)

## Crossover

In [5]:
def crossover(parent1, parent2, child1, child2):
    child1.gap_positions = []
    child2.gap_positions = []
    
    for i in range(len(parent1.gap_positions)):
        gaps1 = parent1.gap_positions[i]
        gaps2 = parent2.gap_positions[i]
        
        # Only do crossover if both parents have gaps
        if len(gaps1) > 0 and len(gaps2) > 0:
            min_len = min(len(gaps1), len(gaps2))
            split = random.randint(0, min_len)
            
            child1.gap_positions.append(sorted(gaps1[:split] + gaps2[split:]))
            child2.gap_positions.append(sorted(gaps2[:split] + gaps1[split:]))
        else:
            # If we one of the parents doesn't have gaps, we switch them for children
            child1.gap_positions.append(gaps2[:])
            child2.gap_positions.append(gaps1[:])
    
    child1.code = child1.build_alignment()
    child2.code = child2.build_alignment()

In [ ]:
def mutation(child, mutation_prob):
    # Go through each sequence of alignment  
    for i in range(len(child.gap_positions)):

        if random.random() < mutation_prob:
            seq_len = len(child.original_sequences[i])
            current_gaps = child.gap_positions[i]
            
            # Randomly choose to insert, remove, or move a gap
            mutation_type = random.choice(['add', 'remove', 'move'])
            
            # Add a gap at randomomly chosen position
            if mutation_type == 'add' or len(current_gaps) == 0:
                if len(current_gaps) < child.max_gaps:
                    total_len = seq_len + len(current_gaps)
                    available = [p for p in range(total_len + 1) if p not in current_gaps]
                    if available:
                        new_gap = random.choice(available)
                        current_gaps.append(new_gap)
                        child.gap_positions[i] = sorted(current_gaps)
            # Remove a random gap
            elif mutation_type == 'remove' and len(current_gaps) > 0:
                current_gaps.pop(random.randint(0, len(current_gaps) - 1))
            
            # Switch position of a gap
            elif mutation_type == 'move' and len(current_gaps) > 0:
                old_gap_idx = random.randint(0, len(current_gaps) - 1)
                current_gaps.pop(old_gap_idx)
                total_len = seq_len + len(current_gaps)
                available = [p for p in range(total_len + 1) if p not in current_gaps]
                if available:
                    new_gap = random.choice(available)
                    current_gaps.append(new_gap)
                    child.gap_positions[i] = sorted(current_gaps)
    
    # Rebuild alignment from modified gap positions
    child.code = child.build_alignment()

## Main GA Function

In [7]:
def ga_msa(sequences, population_size=50, num_generations=100, 
           tournament_size=5, mutation_prob=0.1, elitism_size=5, max_gaps=5):
    
    assert population_size >= 2
    
    population = [Individual(sequences, max_gaps) for _ in range(population_size)]
    new_population = [Individual(sequences, max_gaps) for _ in range(population_size)]

    if elitism_size % 2 != population_size % 2:
        elitism_size += 1
    
    best_fitnesses = []
    avg_fitnesses = []
    
    for generation in range(num_generations):
        population.sort(key=lambda x: x.fitness, reverse=True)
        
        best_fitnesses.append(population[0].fitness)
        avg_fitnesses.append(sum(ind.fitness for ind in population) / len(population))
        
        # if generation % 10 == 0:
        #     print(f"Generation {generation}: Best = {population[0].fitness:.2f}, Avg = {avg_fitnesses[-1]:.2f}")
        
        new_population[:elitism_size] = deepcopy(population[:elitism_size])
        
        for i in range(elitism_size, population_size, 2):
            parent1 = selection(population, tournament_size)
            
            # Temporarily set parent1 fitness to avoid selecting it again
            tmp, parent1.fitness = parent1.fitness, float('-inf')
            parent2 = selection(population, tournament_size)
            parent1.fitness = tmp
            
            crossover(parent1, parent2, new_population[i], new_population[i+1])
            
            mutation(new_population[i], mutation_prob)
            mutation(new_population[i+1], mutation_prob)
            
            new_population[i].calc_fitness()
            new_population[i+1].calc_fitness()
        
        population = deepcopy(new_population)
    
    best_individual = max(population, key=lambda x: x.fitness)
    
    return best_individual, best_fitnesses

## Test Example

In [ ]:
# test_seq = ['PAWHE', 'HEAWY', 'AWHE']

# for i, seq in enumerate(test_seq):
#     print(f"  {i+1}. {seq}")
# print()

# best, fitnesses = ga_msa(
#     sequences=test_seq,
#     population_size=50,
#     num_generations=100,
#     tournament_size=5,
#     mutation_prob=0.1,
#     elitism_size=10,
#     max_gaps=3
# )
# print_individual(best)

  1. PAWHE
  2. HEAWY
  3. AWHE

Score: 10.0
  -PAWHE
  HEAWY-
  --AWHE


In [ ]:
# test_seq2 = ['ACT', 'AT', 'ACT']

# for i, seq in enumerate(test_seq2):
#     print(f"  {i+1}. {seq}")
# print()

# best2, fitnesses2 = ga_msa(
#     sequences=test_seq2,
#     population_size=100,
#     num_generations=50,
#     tournament_size=10,
#     mutation_prob=0.1,
#     elitism_size=10,
#     max_gaps=3
# )

# print_individual(best2)

  1. ACT
  2. AT
  3. ACT

Score: 16.0
  ACT-
  A-T-
  ACT-
